In [1]:
import pandas as pd
from pathlib import Path

diseases = [
    "Acne",
    "Actinic_Keratosis",
    "Benign_tumors",
    "Bullous",
    "Candidiasis",
    "DrugEruption",
    "Eczema",
    "Infestations_Bites",
    "Lichen",
    "Lupus",
    "Moles",
    "Psoriasis",
    "Rosacea",
    "Seborrh_Keratoses",
    "SkinCancer",
    "Sun_Sunlight_Damage",
    "Tinea",
    "Unknown_Normal",
    "Vascular_Tumors",
    "Vasculitis",
    "Vitiligo",
    "Warts",
]

disease_map = {
    "Acne": "Acne",
    "Actinic_Keratosis": "Actinic keratosis",
    "Benign_tumors": "Benign skin tumors",
    "Bullous": "Bullous skin diseases",
    "Candidiasis": "Cutaneous candidiasis",
    "DrugEruption": "Drug eruption",
    "Eczema": "Eczema",
    "Infestations_Bites": "Infestations and bites",
    "Lichen": "Lichen skin diseases",
    "Lupus": "Cutaneous lupus",
    "Moles": "Moles / Nevi",
    "Psoriasis": "Psoriasis",
    "Rosacea": "Rosacea",
    "Seborrh_Keratoses": "Seborrheic keratosis",
    "SkinCancer": "Skin cancer",
    "Sun_Sunlight_Damage": "Sun damage",
    "Tinea": "Tinea / Ringworm",
    "Unknown_Normal": "Normal skin",
    "Vascular_Tumors": "Vascular skin tumors",
    "Vasculitis": "Cutaneous vasculitis",
    "Vitiligo": "Vitiligo",
    "Warts": "Warts",
}

df = pd.DataFrame({
    "model_label": diseases,
    "disease": [disease_map[d] for d in diseases]
})

output_path = Path(
    "/home/jordan/Backup HDD/AI_Machine Learning/DermSight/llm/data/skin_disease_list.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved to {output_path}")
print(df)

Saved to /home/jordan/Backup HDD/AI_Machine Learning/DermSight/llm/data/skin_disease_list.csv
            model_label                 disease
0                  Acne                    Acne
1     Actinic_Keratosis       Actinic keratosis
2         Benign_tumors      Benign skin tumors
3               Bullous   Bullous skin diseases
4           Candidiasis   Cutaneous candidiasis
5          DrugEruption           Drug eruption
6                Eczema                  Eczema
7    Infestations_Bites  Infestations and bites
8                Lichen    Lichen skin diseases
9                 Lupus         Cutaneous lupus
10                Moles            Moles / Nevi
11            Psoriasis               Psoriasis
12              Rosacea                 Rosacea
13    Seborrh_Keratoses    Seborrheic keratosis
14           SkinCancer             Skin cancer
15  Sun_Sunlight_Damage              Sun damage
16                Tinea        Tinea / Ringworm
17       Unknown_Normal             Normal

In [3]:
import pandas as pd
from pathlib import Path


INPUT_PATH = Path("/home/jordan/Backup HDD/AI_Machine Learning/DermSight/llm/data/skin_knowledge_expanded.csv")
OUTPUT_PATH = Path("/home/jordan/Backup HDD/AI_Machine Learning/DermSight/llm/data/skin_knowledge_serving.csv")


def safe_text(value, default=""):
    if pd.isna(value):
        return default

    text = str(value).strip()

    if not text or text.lower() in {"nan", "none", "null"}:
        return default

    return text


def join_parts(*parts):
    cleaned = []

    for part in parts:
        text = safe_text(part)

        if text:
            cleaned.append(text)

    return "; ".join(cleaned)


def build_simple_explanation(disease: str, description: str) -> str:
    if description:
        # Ambil pendek saja agar tidak terlalu panjang untuk prompt.
        sentences = description.split(".")
        short_description = ".".join(sentences[:2]).strip()

        if short_description:
            return short_description + "."

    return (
        f"{disease} adalah kondisi kulit yang perlu dipahami berdasarkan tanda "
        "yang muncul pada kulit. Informasi ini bersifat edukasi dan bukan diagnosis dokter."
    )


def build_safe_actions(treatment: str, prevention: str) -> str:
    base_actions = (
        "Amati perubahan pada kulit; "
        "Jaga kebersihan area kulit; "
        "Hindari menggaruk atau memencet; "
        "Periksa ke puskesmas, pustu, klinik, posyandu, atau tenaga kesehatan jika keluhan memburuk"
    )

    extra_parts = []

    if treatment:
        extra_parts.append(
            f"Ikuti perawatan aman sesuai arahan tenaga kesehatan jika diperlukan: {treatment}"
        )

    if prevention:
        extra_parts.append(
            f"Lakukan pencegahan sesuai informasi knowledge base: {prevention}"
        )

    if extra_parts:
        return join_parts(base_actions, *extra_parts)

    return base_actions


def build_avoid() -> str:
    return (
        "Jangan memakai obat keras tanpa arahan tenaga kesehatan; "
        "Jangan menyimpulkan diagnosis hanya dari AI; "
        "Jangan menunda pemeriksaan jika keluhan memburuk; "
        "Jangan membeli atau menggunakan obat resep tanpa pemeriksaan tenaga kesehatan"
    )


def main():
    df = pd.read_csv(INPUT_PATH)

    rows = []

    for _, row in df.iterrows():
        model_label = safe_text(row.get("model_label"))
        disease = safe_text(row.get("disease"))
        description = safe_text(row.get("description"))
        symptoms = safe_text(row.get("symptoms"))
        treatment = safe_text(row.get("treatment"))
        prevention = safe_text(row.get("prevention"))
        doctor_advice = safe_text(row.get("doctor_advice"))
        source_url = safe_text(row.get("source_url"))

        simple_explanation = build_simple_explanation(
            disease=disease,
            description=description,
        )

        safe_actions = build_safe_actions(
            treatment=treatment,
            prevention=prevention,
        )

        avoid = build_avoid()

        rows.append({
            "model_label": model_label,
            "disease": disease,
            "simple_explanation": simple_explanation,
            "common_signs": symptoms,
            "safe_actions": safe_actions,
            "avoid": avoid,
            "when_to_seek_help": doctor_advice,
            "source_url": source_url,
        })

    output_df = pd.DataFrame(rows)

    for col in output_df.columns:
        output_df[col] = (
            output_df[col]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    print(f"Saved serving knowledge to: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Saved serving knowledge to: /home/jordan/Backup HDD/AI_Machine Learning/DermSight/llm/data/skin_knowledge_serving.csv
